In [15]:
import pandas as pd
prev_agg = pd.read_csv("prev_agg.csv")
df_current_imputed = pd.read_csv("df_current_imputed.csv")
application_data = pd.read_csv("application_data.csv")

In [16]:
print("prev_agg shape:", prev_agg.shape)
print("X_current_clean shape:", df_current_imputed.shape)
print("application_data shape:", application_data.shape)

print("prev_agg columns:", prev_agg.columns[:10].tolist())
print("X_current_clean columns:", df_current_imputed.columns[:10].tolist())
print("application_data columns:", application_data.columns[:10].tolist())

prev_agg shape: (291057, 139)
X_current_clean shape: (307511, 109)
application_data shape: (307511, 122)
prev_agg columns: ['SK_ID_CURR', 'prev_count', 'prev_NO_ACTUAL_LOAN_AMOUNT_ratio', 'prev_REVOLVING_CREDIT_LIMIT_RECORD_ratio', 'prev_PREV_PAYMENT_PLAN_MISSING_ratio', 'prev_CREDIT_APPLICATION_RATIO_DEFINED_ratio', 'prev_RATE_DOWN_PAYMENT_MISSING_ratio', 'prev_prev_approval_stage_available_ratio', 'prev_FLAG_LAST_APPL_PER_CONTRACT_ratio', 'prev_NFLAG_LAST_APPL_IN_DAY_ratio']
X_current_clean columns: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'NAME_TYPE_SUITE']
application_data columns: ['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY']


In [17]:
# 2. 以 current 为准 merge previous
final_df = df_current_imputed.merge(
    prev_agg,
    on="SK_ID_CURR",
    how="left"
)

print("final_df shape before filling:", final_df.shape)
final_df.head(10)

final_df shape before filling: (307511, 247)


,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,...,prev_NAME_SELLER_INDUSTRY_nunique,prev_NAME_YIELD_GROUP_mode,prev_NAME_YIELD_GROUP_mode_ratio,prev_NAME_YIELD_GROUP_nunique,prev_CODE_REJECT_REASON_mode,prev_CODE_REJECT_REASON_mode_ratio,prev_CODE_REJECT_REASON_nunique,prev_NAME_GOODS_CATEGORY_mode,prev_NAME_GOODS_CATEGORY_mode_ratio,prev_NAME_GOODS_CATEGORY_nunique
0,Cash loans,M,0.0,1.0,0.0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,...,1.0,low_normal,1.000000,1.0,XAP,1.000000,1.0,Vehicles,1.000000,1.0
1,Cash loans,F,0.0,0.0,0.0,270000.0,1293502.5,35698.5,1129500.0,Family,...,3.0,middle,0.666667,2.0,XAP,1.000000,1.0,Consumer Electronics,0.333333,3.0
2,Revolving loans,M,1.0,1.0,0.0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,...,1.0,middle,1.000000,1.0,XAP,1.000000,1.0,Mobile,1.000000,1.0
3,Cash loans,F,0.0,1.0,0.0,135000.0,312682.5,29686.5,297000.0,Unaccompanied,...,3.0,XNA,0.444444,4.0,XAP,0.888889,2.0,XNA,0.777778,3.0
4,Cash loans,M,0.0,1.0,0.0,121500.0,513000.0,21865.5,513000.0,Unaccompanied,...,3.0,high,0.500000,2.0,XAP,1.000000,1.0,XNA,0.666667,2.0
5,Cash loans,M,0.0,1.0,0.0,99000.0,490495.5,27517.5,454500.0,"Spouse, partner",...,3.0,low_normal,0.600000,3.0,XAP,1.000000,1.0,XNA,0.400000,4.0
6,Cash loans,F,1.0,1.0,1.0,171000.0,1560726.0,41301.0,1395000.0,Unaccompanied,...,2.0,middle,0.428571,4.0,XAP,1.000000,1.0,Audio/Video,0.285714,4.0
7,Cash loans,M,1.0,1.0,0.0,360000.0,1530000.0,42075.0,1530000.0,Unaccompanied,...,1.0,low_action,1.000000,1.0,XAP,1.000000,1.0,Furniture,1.000000,1.0
8,Cash loans,F,0.0,1.0,0.0,112500.0,1019610.0,33826.5,913500.0,Children,...,2.0,XNA,0.500000,3.0,XAP,0.750000,2.0,XNA,0.750000,2.0
9,Revolving loans,M,0.0,1.0,0.0,135000.0,405000.0,20250.0,405000.0,Unaccompanied,...,2.0,high,0.750000,2.0,XAP,1.000000,1.0,XNA,0.750000,2.0


In [18]:
# Step 3. Mark rows with no previous records
no_prev_mask = final_df["prev_count"].isna()

print("Rows with no previous:", no_prev_mask.sum())
print("No previous ratio:", no_prev_mask.mean())

# Only prev_count should be filled with 0
final_df.loc[no_prev_mask, "prev_count"] = 0

print("prev_count missing after fill:", final_df["prev_count"].isna().sum())
print(final_df["prev_count"].describe())

Rows with no previous: 16454
No previous ratio: 0.05350702901684818
prev_count missing after fill: 0
count    307511.000000
mean          4.597237
std           4.180015
min           0.000000
25%           2.000000
50%           3.000000
75%           6.000000
max          73.000000
Name: prev_count, dtype: float64


In [19]:
# Step 4. Check missing values after merge

missing_summary = pd.DataFrame({
    "column": final_df.columns,
    "missing_count": final_df.isna().sum().values,
    "missing_rate": final_df.isna().mean().values
})

missing_summary = missing_summary.sort_values(
    by="missing_rate",
    ascending=False
)

missing_summary.head(30)

,column,missing_count,missing_rate
202,prev_prev_termination_gap_min,40683,0.132298
201,prev_prev_termination_gap_median,40683,0.132298
200,prev_prev_termination_gap_mean,40683,0.132298
203,prev_prev_termination_gap_max,40683,0.132298
194,prev_DAYS_TERMINATION_CLEAN_min,40497,0.131693
193,prev_DAYS_TERMINATION_CLEAN_median,40497,0.131693
192,prev_DAYS_TERMINATION_CLEAN_mean,40497,0.131693
195,prev_DAYS_TERMINATION_CLEAN_max,40497,0.131693
196,prev_prev_due_period_mean,38399,0.124870
198,prev_prev_due_period_min,38399,0.124870


# Add has_previos indicator

In [21]:
# Step 5. Add has_previous indicator
final_df["has_previous"] = (final_df["prev_count"] > 0).astype(int)

print(final_df["has_previous"].value_counts())
print(final_df["has_previous"].mean())

has_previous
1    291057
0     16454
Name: count, dtype: int64
0.9464929709831518


# FINAL IMPUTATION

In [22]:
# Step 6. Separate previous feature columns by type

prev_cols = [col for col in prev_agg.columns if col != "SK_ID_CURR"]

prev_mode_cols = [
    col for col in prev_cols
    if col.endswith("_mode")
]

prev_nunique_cols = [
    col for col in prev_cols
    if col.endswith("_nunique")
]

prev_ratio_cols = [
    col for col in prev_cols
    if col.endswith("_ratio")
]

prev_ever_cols = [
    col for col in prev_cols
    if col.endswith("_ever")
]

prev_stat_cols = [
    col for col in prev_cols
    if col.endswith(("_mean", "_median", "_min", "_max"))
]

print("mode cols:", len(prev_mode_cols))
print("nunique cols:", len(prev_nunique_cols))
print("ratio cols:", len(prev_ratio_cols))
print("ever cols:", len(prev_ever_cols))
print("stat cols:", len(prev_stat_cols))

covered_cols = (
    set(prev_mode_cols)
    | set(prev_nunique_cols)
    | set(prev_ratio_cols)
    | set(prev_ever_cols)
    | set(prev_stat_cols)
    | {"prev_count"}
)

uncovered_prev_cols = [col for col in prev_cols if col not in covered_cols]

print("uncovered previous cols:", len(uncovered_prev_cols))
print(uncovered_prev_cols)

mode cols: 13
nunique cols: 13
ratio cols: 32
ever cols: 19
stat cols: 60
uncovered previous cols: 0
[]


In [23]:
# Step 7. Impute non-continuous previous features

# 1. mode columns: categorical mode
final_df[prev_mode_cols] = final_df[prev_mode_cols].fillna("No previous")

# 2. nunique columns: no previous means 0 unique categories
final_df[prev_nunique_cols] = final_df[prev_nunique_cols].fillna(0)

# 3. ratio columns: no previous / no occurrence means ratio 0
final_df[prev_ratio_cols] = final_df[prev_ratio_cols].fillna(0)

# 4. ever columns: no previous / never happened means 0
final_df[prev_ever_cols] = final_df[prev_ever_cols].fillna(0)

print("Missing after this step:")
print("mode:", final_df[prev_mode_cols].isna().sum().sum())
print("nunique:", final_df[prev_nunique_cols].isna().sum().sum())
print("ratio:", final_df[prev_ratio_cols].isna().sum().sum())
print("ever:", final_df[prev_ever_cols].isna().sum().sum())

Missing after this step:
mode: 0
nunique: 0
ratio: 0
ever: 0


In [24]:
# Step 8. Impute continuous/stat previous features

# 1. Add missing indicators for stat columns
for col in prev_stat_cols:
    final_df[col + "_missing"] = final_df[col].isna().astype(int)

# 2. Fill stat columns with median
for col in prev_stat_cols:
    median_value = final_df[col].median()
    final_df[col] = final_df[col].fillna(median_value)

# 3. Check missing
print("Missing in stat cols after imputation:", final_df[prev_stat_cols].isna().sum().sum())

# 4. Check how many missing indicators were added
prev_stat_missing_cols = [col + "_missing" for col in prev_stat_cols]
print("Number of stat missing indicators added:", len(prev_stat_missing_cols))
print("Missing indicators shape:", final_df[prev_stat_missing_cols].shape)

C:\Users\rmoch\AppData\Local\Temp\ipykernel_50072\2127003251.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_df[col + "_missing"] = final_df[col].isna().astype(int)
C:\Users\rmoch\AppData\Local\Temp\ipykernel_50072\2127003251.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_df[col + "_missing"] = final_df[col].isna().astype(int)
C:\Users\rmoch\AppData\Local\Temp\ipykernel_50072\2127003251.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times,

Missing in stat cols after imputation: 0
Number of stat missing indicators added: 60
Missing indicators shape: (307511, 60)


In [25]:
print("final_df shape:", final_df.shape)

prev_stat_missing_cols = [col + "_missing" for col in prev_stat_cols]

print("Number of stat missing indicators:", len(prev_stat_missing_cols))
print("Missing in stat cols:", final_df[prev_stat_cols].isna().sum().sum())
print("Missing in stat missing indicators:", final_df[prev_stat_missing_cols].isna().sum().sum())

final_df shape: (307511, 308)
Number of stat missing indicators: 60
Missing in stat cols: 0
Missing in stat missing indicators: 0


In [26]:
# Step 9. Check remaining missing values in the whole final_df

total_missing = final_df.isna().sum().sum()
cols_with_missing = (final_df.isna().sum() > 0).sum()

print("Total missing values:", total_missing)
print("Columns with missing:", cols_with_missing)

remaining_missing = pd.DataFrame({
    "column": final_df.columns,
    "missing_count": final_df.isna().sum().values,
    "missing_rate": final_df.isna().mean().values
}).query("missing_count > 0").sort_values("missing_rate", ascending=False)

remaining_missing.head(30)

Total missing values: 0
Columns with missing: 0


,column,missing_count,missing_rate


# FINAL CHECK

In [27]:
# Step 10. Basic overview of final_df

print("Final shape:", final_df.shape)
print("\nColumn types:")
print(final_df.dtypes.value_counts())

print("\nTotal missing values:", final_df.isna().sum().sum())
print("Columns with missing:", (final_df.isna().sum() > 0).sum())

print("\nDuplicated SK_ID_CURR:", final_df["SK_ID_CURR"].duplicated().sum())

print("\nTARGET distribution:")
print(final_df["TARGET"].value_counts())

print("\nTARGET ratio:")
print(final_df["TARGET"].value_counts(normalize=True))

Final shape: (307511, 308)

Column types:
float64    188
int64       93
object      27
Name: count, dtype: int64

Total missing values: 0
Columns with missing: 0

Duplicated SK_ID_CURR: 0

TARGET distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64

TARGET ratio:
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


In [28]:
# Save final merged + imputed dataset
final_df.to_csv("final_df.csv", index=False)